# Unconstraining Analysis

Compares **Naive / EM / PD(τ)** unconstrained demand estimates against the oracle ground truth.

**Pipeline:**
1. `batch_unconstrain.py` reads `flights_truncated.csv` (no oracle) and runs a rolling 365-day window per cabin per flight.
2. This notebook loads the parquet results, merges in oracle truth from `flights_latent.csv`, and reports WAPE / MAE / lift.

**Run the batch job first** (cell below), then execute the rest.

In [ ]:
import subprocess, sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, os.path.join('..', 'unconstraining'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#0e1117','axes.facecolor':'#161b22','axes.edgecolor':'#30363d',
    'axes.labelcolor':'#c9d1d9','axes.titlecolor':'#f0f6fc','xtick.color':'#8b949e',
    'ytick.color':'#8b949e','text.color':'#c9d1d9','grid.color':'#21262d',
    'grid.linewidth':0.6,'legend.facecolor':'#161b22','legend.edgecolor':'#30363d',
    'legend.labelcolor':'#c9d1d9','font.size':11,'axes.titlesize':13,'axes.titleweight':'bold',
})
C = dict(
    oracle='#c9d1d9', observed='#3fb950', naive='#8b949e',
    em='#58a6ff', censored='#f85149', capacity='#d29922',
    first='#bc8cff', business='#58a6ff', premium_eco='#3fb950', economy='#79c0ff',
    pd01='#ffa657', pd03='#d29922', pd05='#f85149', pd07='#ff7b72', pd09='#bc8cff',
)
CABINS   = ['first', 'business', 'premium_eco', 'economy']
PD_TAUS  = [0.1, 0.3, 0.5, 0.7, 0.9]
PREFIXES = ['naive', 'em'] + [f'pd{str(t).replace(".","")}' for t in PD_TAUS]
LABELS   = {'naive':'Naive','em':'EM'} | {f'pd{str(t).replace(".","")}':f'PD(τ={t})' for t in PD_TAUS}
ROOT     = os.path.join('..', 'data')
RES_DIR  = os.path.join('..', 'unconstraining', 'results')
FIG_DIR  = os.path.join('..', 'unconstraining', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print('Setup OK')

In [ ]:
# ── Run unconstraining batch job ──────────────────────────────────────────────
# Skip if parquets already exist with per-cabin columns
parquets = [f for f in os.listdir(RES_DIR) if f.endswith('.parquet')]
needs_run = True
if parquets:
    test = pd.read_parquet(os.path.join(RES_DIR, parquets[0]))
    needs_run = 'em_first_est' not in test.columns

if needs_run:
    print('Running batch_unconstrain.py (this takes a few minutes)...')
    result = subprocess.run(
        [sys.executable, os.path.join('..', 'unconstraining', 'batch_unconstrain.py')],
        capture_output=True, text=True
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
else:
    print('Results already up-to-date — skipping batch run.')

In [ ]:
# ── Load all per-flight parquets + merge oracle ───────────────────────────────
parquets = sorted(f for f in os.listdir(RES_DIR) if f.endswith('.parquet'))
frames   = [pd.read_parquet(os.path.join(RES_DIR, p)) for p in parquets]
df_res   = pd.concat(frames, ignore_index=True)
df_res['date'] = pd.to_datetime(df_res['date'])

oracle_cols = ['date','flight_od'] + [f'oracle_{c}_pax' for c in CABINS] + ['latent_seats']
df_lat = pd.read_csv(os.path.join(ROOT, 'flights_latent.csv'), parse_dates=['date'],
                     usecols=oracle_cols)

df = df_res.merge(df_lat, on=['date','flight_od'], how='left')

print(f'Loaded: {len(df):,} rows | {df.flight_od.nunique()} flights')
print(f'Columns: {[c for c in df.columns if "_est" in c][:8]} ...')

---
## 1 — WAPE & MAE by model and cabin

Computed only on **censored cabin-days** — days where the cabin sold out and the unconstrainer had something to recover.

In [ ]:
def wape(true, est):
    mask = true > 0
    return np.abs(est[mask] - true[mask]).sum() / true[mask].sum()

def mae(true, est):
    return np.abs(est - true).mean()

rows = []
for cab in CABINS:
    cab_cens = df[f'{cab}_pax'] >= df[f'{cab}_capacity']
    sub = df[cab_cens].copy()
    truth = sub[f'oracle_{cab}_pax'].values
    for prefix in PREFIXES:
        est_col = f'{prefix}_{cab}_est'
        if est_col not in sub.columns: continue
        est = sub[est_col].values
        rows.append({
            'cabin': cab, 'model': LABELS[prefix],
            'WAPE': round(wape(truth, est), 4),
            'MAE':  round(mae(truth, est), 2),
            'n_cens': int(cab_cens.sum()),
        })

metrics = pd.DataFrame(rows)
pivot_wape = metrics.pivot(index='model', columns='cabin', values='WAPE')
pivot_mae  = metrics.pivot(index='model', columns='cabin', values='MAE')

print('WAPE (lower = better)'); print(pivot_wape.round(4).to_string())
print('\nMAE (lower = better)'); print(pivot_mae.round(2).to_string())

---
## 2 — τ sensitivity for PD models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PD τ sensitivity — WAPE and MAE on censored cabin-days', fontsize=13)

tau_labels = [f'PD(τ={t})' for t in PD_TAUS]

for ax, metric_name, pivot in zip(axes, ['WAPE', 'MAE'], [pivot_wape, pivot_mae]):
    pd_rows = pivot.loc[[l for l in tau_labels if l in pivot.index]]
    for cab in CABINS:
        if cab not in pd_rows.columns: continue
        ax.plot(PD_TAUS, pd_rows[cab].values,
                marker='o', ms=6, lw=2, color=C[cab],
                label=cab.replace('_',' ').title())
    ax.set_xlabel('τ'); ax.set_ylabel(metric_name)
    ax.set_title(f'{metric_name} vs τ — all cabins')
    ax.legend(fontsize=9); ax.grid(True)
    ax.set_xticks(PD_TAUS)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig1_tau_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 3 — WAPE heatmap: flight × cabin

In [ ]:
# Best model per cabin (lowest WAPE across EM + PD)
best_prefix = {}
for cab in CABINS:
    cab_cens = df[f'{cab}_pax'] >= df[f'{cab}_capacity']
    sub = df[cab_cens]
    truth = sub[f'oracle_{cab}_pax'].values
    best_w, best_p = 1e9, 'em'
    for prefix in PREFIXES:
        ec = f'{prefix}_{cab}_est'
        if ec not in sub.columns: continue
        w = wape(truth, sub[ec].values)
        if w < best_w: best_w, best_p = w, prefix
    best_prefix[cab] = best_p

# WAPE heatmap: flight (rows) × cabin (cols), using best model per cabin
flights = sorted(df.flight_od.unique())
heatmap = pd.DataFrame(index=flights, columns=CABINS, dtype=float)

for fid in flights:
    flt = df[df.flight_od == fid]
    for cab in CABINS:
        prefix = best_prefix[cab]
        cab_cens = flt[f'{cab}_pax'] >= flt[f'{cab}_capacity']
        sub = flt[cab_cens]
        if len(sub) == 0: continue
        truth = sub[f'oracle_{cab}_pax'].values
        est   = sub[f'{prefix}_{cab}_est'].values
        heatmap.loc[fid, cab] = wape(truth, est)

fig, ax = plt.subplots(figsize=(10, max(4, len(flights) * 0.55)))
data = heatmap.values.astype(float)
im = ax.imshow(data, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.3, interpolation='nearest')
ax.set_xticks(range(len(CABINS)))
ax.set_xticklabels([c.replace('_',' ').title() for c in CABINS])
ax.set_yticks(range(len(flights)))
ax.set_yticklabels([f.replace('->','→') for f in flights], fontsize=9)
for i, fid in enumerate(flights):
    for j, cab in enumerate(CABINS):
        v = data[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=8, color='#0e1117' if v < 0.2 else '#f0f6fc')
plt.colorbar(im, ax=ax, label='WAPE (best model per cabin)')
ax.set_title(f'WAPE heatmap — best model per cabin ({"|".join(best_prefix.values())})')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig2_wape_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4 — Time series: observed vs unconstrained vs oracle

In [ ]:
import matplotlib.dates as mdates

# Pick the flight with the most censored economy days
cens_by_flight = (
    df.assign(eco_cens=lambda x: x['economy_pax'] >= x['economy_capacity'])
    .groupby('flight_od')['eco_cens'].sum()
    .sort_values(ascending=False)
)
ROUTE = cens_by_flight.index[0]
YEAR  = df[df.flight_od == ROUTE]['date'].dt.year.max() - 1

flt  = df[(df.flight_od == ROUTE) & (df.date.dt.year == YEAR)].sort_values('date')

fig, axes = plt.subplots(len(CABINS), 1, figsize=(15, 3.5 * len(CABINS)), sharex=True)
fig.suptitle(f'{ROUTE.replace("->","→")} — Observed vs Unconstrained vs Oracle ({YEAR})', fontsize=13)

for ax, cab in zip(axes, CABINS):
    cap = int(flt[f'{cab}_capacity'].iloc[0])
    oracle_col = f'oracle_{cab}_pax'

    # Oracle (ground truth)
    ax.plot(flt.date, flt[oracle_col], color=C['oracle'],
            lw=1.2, alpha=0.7, label='Oracle', zorder=2)
    # EM estimate
    ax.plot(flt.date, flt[f'em_{cab}_est'], color=C['em'],
            lw=1.5, label='EM', zorder=3)
    # Best PD (τ=0.5)
    ax.plot(flt.date, flt[f'pd05_{cab}_est'], color=C['pd05'],
            lw=1.5, ls='--', label='PD(τ=0.5)', zorder=3)
    # Observed (what RM sees)
    ax.fill_between(flt.date, 0, flt[f'{cab}_pax'],
                    color=C['observed'], alpha=0.25, label='Observed')
    # Capacity ceiling
    ax.axhline(cap, color=C['capacity'], lw=1, ls=':', label=f'Capacity ({cap})')

    ax.set_ylabel('Passengers', fontsize=9)
    ax.set_title(f'{cab.replace("_"," ").title()} cabin', fontsize=10)
    ax.legend(fontsize=8, loc='upper left', ncol=5)
    ax.grid(True, axis='y')

axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig3_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Route: {ROUTE}  |  Year: {YEAR}')

---
## 5 — Lift distribution: how much does each model add above observed?

In [ ]:
fig, axes = plt.subplots(1, len(CABINS), figsize=(16, 4), sharey=False)
fig.suptitle('Lift distribution (unconstrained − observed) on censored cabin-days', fontsize=13)

plot_prefixes = ['naive', 'em', 'pd03', 'pd05', 'pd07']

for ax, cab in zip(axes, CABINS):
    cab_cens = df[f'{cab}_pax'] >= df[f'{cab}_capacity']
    sub = df[cab_cens]
    for prefix in plot_prefixes:
        col = f'{prefix}_{cab}_est'
        if col not in sub.columns: continue
        lift = sub[col].values - sub[f'{cab}_pax'].values
        ax.hist(lift, bins=40, alpha=0.55, color=C.get(prefix, '#8b949e'),
                label=LABELS[prefix], density=True)
    ax.axvline(0, color='#c9d1d9', lw=1.2, ls='--')
    ax.set_title(cab.replace('_',' ').title())
    ax.set_xlabel('Lift (pax)')
    if ax == axes[0]: ax.set_ylabel('Density')
    ax.legend(fontsize=7); ax.grid(True, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig4_lift_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6 — Oracle recovery: estimated vs true demand (censored days only)

In [ ]:
fig, axes = plt.subplots(2, len(CABINS), figsize=(16, 8))
fig.suptitle('Oracle recovery — estimated vs true demand (EM top, PD(τ=0.5) bottom)', fontsize=13)

for col_idx, cab in enumerate(CABINS):
    cab_cens = df[f'{cab}_pax'] >= df[f'{cab}_capacity']
    sub = df[cab_cens]
    truth = sub[f'oracle_{cab}_pax'].values
    cap   = int(sub[f'{cab}_capacity'].iloc[0])
    lim   = max(truth.max(), sub[f'em_{cab}_est'].values.max()) * 1.05

    for row_idx, prefix in enumerate(['em', 'pd05']):
        ax = axes[row_idx][col_idx]
        est = sub[f'{prefix}_{cab}_est'].values
        ax.scatter(truth, est, s=4, alpha=0.3, color=C.get(prefix, '#58a6ff'), rasterized=True)
        ax.plot([0, lim], [0, lim], color='#c9d1d9', lw=1, ls='--', label='Perfect')
        ax.axvline(cap, color=C['capacity'], lw=0.8, ls=':', alpha=0.6)
        ax.axhline(cap, color=C['capacity'], lw=0.8, ls=':', alpha=0.6)
        ax.set_xlim(0, lim); ax.set_ylim(0, lim)
        if row_idx == len(axes) - 1:
            ax.set_xlabel('Oracle (true demand)')
        if col_idx == 0:
            ax.set_ylabel(f'{LABELS[prefix]}\nEstimated demand')
        ax.set_title(cab.replace('_',' ').title() if row_idx == 0 else '')
        ax.grid(True, alpha=0.3)
        w = wape(truth, est)
        ax.text(0.97, 0.05, f'WAPE={w:.3f}', transform=ax.transAxes,
                ha='right', fontsize=8, color='#c9d1d9')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig5_oracle_recovery.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 7 — Censoring rate vs WAPE: does higher censoring make it harder?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Censoring rate vs WAPE — EM and PD(τ=0.5) per flight × cabin', fontsize=13)

for ax, prefix in zip(axes, ['em', 'pd05']):
    for cab in CABINS:
        points = []
        for fid in df.flight_od.unique():
            flt = df[df.flight_od == fid]
            cab_cens = flt[f'{cab}_pax'] >= flt[f'{cab}_capacity']
            cens_rate = cab_cens.mean()
            if cens_rate == 0: continue
            sub = flt[cab_cens]
            truth = sub[f'oracle_{cab}_pax'].values
            est   = sub[f'{prefix}_{cab}_est'].values
            w = wape(truth, est)
            points.append((cens_rate, w))
        if not points: continue
        xv, yv = zip(*points)
        ax.scatter(xv, yv, s=60, alpha=0.8, color=C[cab],
                   label=cab.replace('_',' ').title())
    ax.set_xlabel('Cabin censoring rate'); ax.set_ylabel('WAPE')
    ax.set_title(LABELS[prefix])
    ax.legend(fontsize=9); ax.grid(True)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig6_censoring_vs_wape.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 8 — Model comparison summary

In [ ]:
# Overall WAPE per model (pooled across all cabins + flights, censored days)
summary_rows = []
for prefix in PREFIXES:
    total_abs_err, total_true = 0.0, 0.0
    total_lift, n = 0.0, 0
    for cab in CABINS:
        cab_cens = df[f'{cab}_pax'] >= df[f'{cab}_capacity']
        sub = df[cab_cens]
        truth = sub[f'oracle_{cab}_pax'].values
        col   = f'{prefix}_{cab}_est'
        if col not in sub.columns: continue
        est = sub[col].values
        total_abs_err += np.abs(est - truth).sum()
        total_true    += truth[truth > 0].sum()
        total_lift    += (est - sub[f'{cab}_pax'].values).sum()
        n             += len(sub)
    summary_rows.append({
        'Model': LABELS[prefix],
        'Overall WAPE': round(total_abs_err / total_true, 4) if total_true > 0 else np.nan,
        'Avg Lift/day':  round(total_lift / n, 2) if n > 0 else np.nan,
        'N censored':    n,
    })

summary = pd.DataFrame(summary_rows).set_index('Model')
print(summary.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(12, 4))
models_plot = [LABELS[p] for p in PREFIXES]
wapes_plot  = [summary.loc[LABELS[p], 'Overall WAPE'] for p in PREFIXES]
colors_bar  = [C.get(p, '#8b949e') for p in PREFIXES]
bars = ax.bar(models_plot, wapes_plot, color=colors_bar, alpha=0.85)
ax.bar_label(bars, fmt='%.4f', fontsize=8, color='#c9d1d9', padding=3)
ax.set_ylabel('Overall WAPE (censored cabin-days)')
ax.set_title('Model comparison — pooled WAPE across all flights & cabins')
ax.set_ylim(0, max(wapes_plot) * 1.2)
plt.xticks(rotation=30, ha='right')
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig7_model_summary.png'), dpi=150, bbox_inches='tight')
plt.show()